# Candle Streaks 01 — Every consecutive five-minute price-sign run

**Historical exploratory research. No arbitrary fixed 10-bar samples.** Walk each asset's entire available five-minute return history sequentially, recording every completed streak (`+1`, `+2`, ... and `−1`, `−2`, ...) and each point where a live streak reaches length `k`. Then test whether the *immediately next nonzero* candle is more or less likely to continue the streak.

The main forecasting controls are **50/50**, empirical unconditional sign frequency, and a **first-order Markov** sign-only model. A streak-length effect is useful only if it beats Markov out of sample. AM-T-HAR-Blend is **not** modified here. [Frozen study protocol](../../discussions/Candle_Streaks/001_chatgpt_candle_streaks_preregistration.md).

## 0 · Environment, reproducibility and synthetic regression tests

In [ ]:
import json, os, sys, subprocess, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from IPython.display import display
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(),'Open notebook within the cloned GitHub repository'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import markov_lift_distributions as vd14
from data_engineering.modeling import candle_streaks as cs
cfg=cs.Specification()
HIST=Path(os.environ['DERIBIT_DATA_ROOT'])/'history'/'price_5m'
assert HIST.is_dir(),f'Cannot locate local Binance price history: {HIST}'
OUT=ROOT/'notebooks'/'Candle_Streaks'/'01';OUT.mkdir(parents=True,exist_ok=True)
START=time.monotonic()
pytest_env=os.environ.copy()
pytest_env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+pytest_env.get('PYTHONPATH','')
chk=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_candle_streaks.py')],
                   cwd=str(ROOT),env=pytest_env,capture_output=True,text=True)
print(chk.stdout[-2500:]);print(chk.stderr[-1000:])
assert chk.returncode==0,'Synthetic tests must pass before analyzing history'
print('Research registration:',json.dumps(cs.registration(),indent=2))

## 1 · Load **all available** contiguous five-minute close-to-close returns

The frozen cutoff is the start of 2026-10-14 UTC; only completed days through 2026-10-13 are included. The existing VD14 loader handles off-grid observations, duplicates and missing-bar return invalidation. `ASSETS` are BTC/ETH primary and SOL/XRP secondary. Missing/insufficient-history assets appear in `excluded`. No fixed rolling window is used.

In [ ]:
raw,excluded=vd14.load_returns(HIST, tuple(cfg.assets),
                               end=pd.Timestamp(cfg.cutoff_utc),min_days=365)
print('Source exclusions:',excluded)
assert {'BTC','ETH'}.issubset(set(raw.underlying.unique())),'BTC and ETH price history required'
assert pd.to_datetime(raw.state_time,utc=True).max()<pd.Timestamp(cfg.cutoff_utc)
print('Loaded asset rows:',raw.groupby('underlying').size().to_dict())
print('Date range:',raw.state_time.min(),raw.state_time.max())
events,runs,quality=cs.build_events_and_runs(raw)
print('Data quality / strict run boundary statistics:')
display(quality)
print('Count of all five-minute observations, consecutive signed opportunities and episode ends:',len(raw),len(events),len(runs))

## 2 · Completed streak distributions and *reached-k* continuation frequencies

A `+3` completed episode is one **maximal** three-up run whose ending is observed. A *reached `+3`* opportunity counts every live run the moment it gets to length 3. The first is a distribution of episode lengths; the second is the causal information available before the next bar. Missing bars right-censor episodes; zero returns end runs but are not up/down. **Both tables preserve all observed streak lengths**, however long.

In [ ]:
# Geometric null uses marginal UP/DOWN/ZERO probabilities of ALL valid bars,
# not sign frequencies conditional on the previous bar being nonzero.
base_rates={(q.asset,sign):float((q.up if sign==1 else q.down)/q.valid_returns)
            for q in quality.itertuples() for sign in (-1,1)}
completed, hazards=cs.distribution_tables(events,runs,base_rates=base_rates)
print('Full observed signed run-length range by asset (not truncated):')
display(runs.groupby('underlying').agg(max_up=('signed_streak','max'),
                                       max_down=('signed_streak','min'),
                                       completed_runs=('completed','sum'),
                                       total_runs=('signed_streak','size')))
for asset in sorted(events.underlying.unique()):
    print('\n',asset,'short-length excerpt (ALL lengths saved below)')
    sample=hazards[(hazards.underlying==asset)&(hazards.length<=12)].copy()
    display(sample[['signed_streak','n_reached','n_next_nonzero','n_continue','n_reverse',
                    'n_zero','n_censored','p_continue_given_next_nonzero','wilson95_low','wilson95_high']].round(4))
    part=completed[completed.underlying==asset]
    fig,ax=plt.subplots(figsize=(9,4))
    for sign,label in [(1,'Up streak'),(-1,'Down streak')]:
        sub=part[part.sign==sign].sort_values('length')
        ax.plot(sub.length,sub.n_completed,marker='.',linestyle='-',label=label)
    ax.set_yscale('symlog',linthresh=1);ax.set_xlabel('Completed streak length in five-minute candles')
    ax.set_ylabel('Count (symlog)');ax.set_title(asset+' — completed sign-run distribution')
    ax.legend();plt.show()
print('Interpretation: Wilson intervals are pointwise/descriptive; adjacent bars and sessions may be dependent.')

## 3 · Independent-sign geometric reference and preserved-zero permutation null

Under an independent 50/50 sign process, the probability of three **specified** up candles is `0.5**3 = 12.5%`, and the probability the NEXT up candle continues a three-up streak remains 50%. For observed data we also calculate sign marginals directly; the approximate geometric expected run count uses observed starts and their empirical same-sign chance. The permutation reference shuffles only nonzero signs, retaining exact zero/missing positions and gaps; it does not preserve clock effects or evolving drift.

In [ ]:
print('Specified 3-up prefix under fair independent signs:',.5**3)
for asset in sorted(events.underlying.unique()):
    g=events[(events.underlying==asset)&events.next_valid]
    p_up=np.mean(g.next_sign==1);p_down=np.mean(g.next_sign==-1);p_zero=np.mean(g.next_sign==0)
    print(asset,'next-bar unconditional observed fractions up/down/zero:',round(p_up,5),round(p_down,5),round(p_zero,5))
permutations=cs.iid_permutation_null(raw,reps=cfg.permutation_reps,seed=cfg.seed)
display(permutations.round(3))

## 4 · Chronological forecasting: does **length** add information beyond the last sign?

Chronological full-day split 60% FIT / 20% VALID / 20% TEST. Every model is fitted on FIT ONLY. Forecasting excludes missing/zero next bars and any target crossing a split boundary. All comparisons are made on exactly the same immediate next-nonzero 5-minute candles. A smoothed probability is estimated for **each actually observed signed length**; an unseen `k` falls back to the Markov baseline, never to an arbitrary clipped length. The log-length model is a smoother alternative.

In [ ]:
labeled,dates=cs.add_chronological_splits(events,cfg.train_fraction,cfg.valid_fraction)
print('Boundaries:',dates)
print('Scorable opportunities:',labeled.groupby(['underlying','split']).scorable.sum())
models={};scores=[];boot=[];nobs=[]
for asset,g in labeled.groupby('underlying',sort=True):
    tr=g.loc[g.split=='train'];va=g.loc[g.split=='valid'];te=g.loc[g.split=='test']
    if not (tr.scorable.sum()>=100 and va.scorable.sum()>=100 and te.scorable.sum()>=100):
        print('Skipping',asset,': inadequate eligible train/valid/test nonzero next-bar pairs')
        continue
    fit,sc,ci,ct=cs.analyze_asset(tr,va,te,pseudo=cfg.length_prior,
          ridge=cfg.logistic_l2,reps=cfg.bootstrap_reps,seed=cfg.seed)
    models[asset]=fit;sc['asset']=asset;ci['asset']=asset;ct['asset']=asset
    scores.append(sc);boot.append(ci);nobs.append(ct)
score_table=pd.concat(scores,ignore_index=True)
bootstrap=pd.concat(boot,ignore_index=True)
counts=pd.concat(nobs,ignore_index=True)
print('Holdout sign forecast scores (lower logloss/Brier is better):')
display(score_table.sort_values(['asset','period','logloss']).round(7))
print('Paired day bootstrap: streak minus each benchmark (negative = better)')
display(bootstrap.loc[bootstrap.period=='test'].sort_values(['asset','model','reference','metric']).round(7))

## 5 · Empirical continuation hazards and economic interpretation checks

The observed geometric null is not inferred from a ten-bar sequence: it's compared with the **entire history of actual runs**. Consider both directions separately. Long streaks have fewer observations and wider uncertainty. The next-nonzero continuation probability is not the same as the unconditional probability for *any* next candle, because flat candles are a third outcome.

In [ ]:
for asset in sorted(events.underlying.unique()):
    sub=hazards[(hazards.underlying==asset)&(hazards.n_next_nonzero>=25)]
    fig,ax=plt.subplots(figsize=(9,4))
    for sign,label in [(1,'Up continuation'),(-1,'Down continuation')]:
        q=sub[sub.sign==sign].sort_values('length')
        ax.plot(q.length,q.p_continue_given_next_nonzero,marker='.',label=label)
    ax.axhline(.5,linestyle='--',label='Independent fair-sign null')
    ax.set_ylim(0,1);ax.set_xlabel('Current consecutive five-minute candles (exact k)')
    ax.set_ylabel('P(next bar continues | next nonzero)')
    ax.set_title(asset+' — empirical live-streak continuation; min 25 observed next signs for plot only')
    ax.legend();plt.show()
    fit=models.get(asset)
    if fit:
        print(asset,'TRAIN-only Markov continuation:',fit['markov'],
              '| signed log-length coefficient:',fit['logistic_coefficients'][2:])
print('The full hazard CSV includes every length, including n<25; plotting threshold is not a data cap.')

## 6 · Frozen test decision and compact research artifacts

For a candidate useful length effect, at least one fixed length-sensitive model must improve TEST **logloss** over both Markov and marginal-sign benchmarks, and do so for BTC and ETH, with 95% paired-day upper bounds below zero. The empirical distributions are valuable whether or not this forecast gate passes. These historical dates have already been researched in other branches, so even a passing result is exploratory.

In [ ]:
decisions=[]
for asset in ('BTC','ETH'):
    bb=bootstrap[(bootstrap.asset==asset)&(bootstrap.period=='test')&(bootstrap.metric=='logloss')]
    options=[]
    for model in ('streak_exact_shrunk','streak_log_duration'):
        candidate=bb[bb.model==model]
        passes=(len(candidate)==2 and (candidate.ci975<0).all())
        options.append({'model':model,'beats_both_benchmarks':bool(passes),
                        'upper95_v_markov':float(candidate.loc[candidate.reference=='markov_sign','ci975'].iloc[0]),
                        'upper95_v_iid':float(candidate.loc[candidate.reference=='iid_empirical','ci975'].iloc[0])})
    decisions.append({'asset':asset,'candidate_models':options})
accepted=any(all(any(z['model']==model and z['beats_both_benchmarks'] for z in d['candidate_models']) for d in decisions)
             for model in ('streak_exact_shrunk','streak_log_duration'))
verdict='HISTORICAL_LENGTH_CANDIDATE' if accepted else 'NO_CONFIRMED_INCREMENTAL_STREAK_SIGNAL'
print('Decision:',verdict)
display(pd.DataFrame(decisions))
output={
 '01_quality.csv':quality,'01_completed_run_lengths.csv':completed,
 '01_reached_length_hazards.csv':hazards,'01_heldout_direction_scores.csv':score_table,
 '01_paired_day_bootstrap.csv':bootstrap,'01_iid_permutation_null.csv':permutations,
 '01_period_counts.csv':counts,
}
for fn,tab in output.items():
    tab.to_csv(OUT/fn,index=False,float_format='%.10g')
serializable={asset:{'p_up':float(f['p_up']),
                        'markov':{str(k):float(v) for k,v in f['markov'].items()},
                        'logistic_coefficients':f['logistic_coefficients'],
                        'training_rows':f['training_rows'],'length_prior':f['pseudo'],
                        'exact_k_levels_fit':len(f['exact']),
                        'max_k_fit':max(k for _,k in f['exact'])} for asset,f in models.items()}
(OUT/'01_fit_summary.json').write_text(json.dumps(serializable,indent=2),encoding='utf-8')
manifest={'module':'Candle_Streaks_01','specification':cs.registration(),
          'source':'Binance spot five-minute close-to-close returns via VD14 load_returns',
          'excluded':excluded,'date_splits':dates,
          'asset_span':quality.to_dict('records'),
          'verdict':verdict,'decisions':decisions,
          'notes':['All observed streak lengths retained; no fixed number of candles per sample.',
                   'Zero returns break runs; binary forecast conditional on next nonzero.',
                   'Gap/file-end right censor, not reversed streaks.',
                   'All models trained on earlier full UTC dates and never refit on evaluation.',
                   'Historical exploratory only; no AM-T-HAR-Blend changes or options trading edge.']}
(OUT/'01_manifest.json').write_text(json.dumps(manifest,indent=2,default=str),encoding='utf-8')
print('Saved',len(output)+2,'aggregate files to',OUT)
print('Elapsed (seconds):',round(time.monotonic()-START,1))

## Interpretation

A long completed streak is **not** proof that the next bar is more likely to reverse. The fair independent-sign process expects increasingly rare long runs even though its next-sign chance is always 50%. The key questions are whether the observed continuation-vs-length curve differs from the independent and first-order Markov references, **and** whether those differences generalize to chronologically later bars. The models score the immediate next **nonzero** five-minute return only; if zero returns are frequent, add a separate three-class forecast in a future registered module. Direction probabilities are not automatically improvements to AM-T-HAR-Blend's return-density forecasts. This project does not change the AM baseline unless a later matched-horizon experiment confirms value.